# Climate Mitigation Potential widget data preparation

## Load libraries

In [1]:
import pandas as pd
import json
import geopandas as gpd
import subprocess


In [2]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data/"

## Load and clean data
Mitigation potential data is stored in a CSV on the [GCS folder](https://console.cloud.google.com/storage/browser/wetlands-gap-map/original_data;tab=objects?inv=1&invt=Ab12FA&prefix=&forceOnObjectsSortingFiltering=false).  
It contains all countries, needs to be filtered to Sahel countries.  


In [3]:
df = pd.read_csv(f'{gcs_path}Mitigation_Country_clean.csv')
sel_cols = ['ISO',
        'Reduce deforestation AVG', 'Reforestation AVG', 'Forest management AVG',
       'Grassland and savanna fire mgmt ',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration'] # Selected columns need to be reviewed for final data
df = df[sel_cols]
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower().str.replace('_avg', '')
df.head()

,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
0,AFG,54.7488,84.1559,100.6039,NaN,NaN,NaN,NaN,NaN
1,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALB,NaN,54.0851,9.6573,NaN,NaN,NaN,NaN,NaN
3,DZA,NaN,126.3656,6.4040,NaN,NaN,NaN,NaN,NaN
4,ASM,NaN,547.1500,0.0000,NaN,NaN,NaN,NaN,NaN


In [4]:
locations = gpd.read_file(f'{gcs_path}locations_simplified.geojson')
locations = locations[locations['type'].isin(['admin_region', 'global'])]
countries = locations[locations['type'] == 'admin_region'][['name', 'code']].copy()

In [5]:
#Filter to Sahel countries
print(f'Initial countries: {len(df)}')
df = df[df['iso'].isin(countries['code'])]
print(f'Countries after filtering: {len(df)}')

Initial countries: 250
Countries after filtering: 26


In [6]:
df.columns

Index(['iso', 'reduce_deforestation', 'reforestation', 'forest_management',
       'grassland_and_savanna_fire_mgmt', 'reduce_peatland_degradation',
       'peatland_restoration', 'reduce_mangrove_conversion',
       'mangrove_restoration'],
      dtype='object')

In [7]:
locations.head(2)

,id,name,type,code,geometry
0,global_sahel,Global (Sahel),global,global_sahel,"POLYGON ((42.94782 10.99465, 42.97027 10.99147..."
1,adminregion_sen,Senegal,admin_region,SEN,"POLYGON ((-17.74987 14.74329, -17.74334 14.693..."


In [8]:
global_data = {'iso':'global_sahel',
               'reduce_deforestation': df['reduce_deforestation'].mean(skipna=True),
               'reforestation': df['reforestation'].mean(skipna=True),
               'forest_management': df['forest_management'].mean(skipna=True),
               'grassland_and_savanna_fire_mgmt': df['grassland_and_savanna_fire_mgmt'].mean(skipna=True),
               'reduce_peatland_degradation': df['reduce_peatland_degradation'].mean(skipna=True),
               'peatland_restoration': df['peatland_restoration'].mean(skipna=True),
               'reduce_mangrove_conversion': df['reduce_mangrove_conversion'].mean(skipna=True),
               'mangrove_restoration': df['mangrove_restoration'].mean(skipna=True)}

df_full = pd.concat([df, pd.DataFrame([global_data])], ignore_index=True)

## Process data.  
Process and reshape the data for match needed format.

In [9]:
# Pivot to long table and create location id column
df_long = df_full.melt(id_vars=['iso'], var_name='intervention', value_name='value')
df_long['location'] = df_long['iso'].apply(lambda x: f'adminregion_{x.lower()}' if x != 'global_sahel' else 'global_sahel')
df_long = df_long[df_long['value'].notna()]
df_long = df_long[['location', 'intervention', 'value']]
df_long

,location,intervention,value
0,adminregion_ben,reduce_deforestation,262.9976
1,adminregion_bfa,reduce_deforestation,115.6229
2,adminregion_cmr,reduce_deforestation,374.3571
3,adminregion_caf,reduce_deforestation,262.3109
4,adminregion_tcd,reduce_deforestation,225.7684
...,...,...,...
208,adminregion_sen,mangrove_restoration,704.0000
209,adminregion_sle,mangrove_restoration,704.0000
210,adminregion_som,mangrove_restoration,704.0000
212,adminregion_sdn,mangrove_restoration,704.0000


In [10]:
intervention_type_map = {'reduce_deforestation':'protection',
 'reforestation':'restoration',
 'forest_management':'protection',
 'grassland_and_savanna_fire_mgmt':'protection',
 'reduce_peatland_degradation':'protection',
 'peatland_restoration':'restoration',
 'reduce_mangrove_conversion':'protection',
 'mangrove_restoration':'restoration'}
ecosystem_type_map = {'reduce_deforestation':'non-wetlands',
 'reforestation':'non-wetlands',
 'forest_management':'non-wetlands',
 'grassland_and_savanna_fire_mgmt':'non-wetlands',
 'reduce_peatland_degradation':'wetlands',
 'peatland_restoration':'wetlands',
 'reduce_mangrove_conversion':'wetlands',
 'mangrove_restoration':'wetlands'}
color_map = {'reduce_deforestation':'#628c5f',
 'reforestation':'#98ed93',
 'forest_management':'#255c21',
 'grassland_and_savanna_fire_mgmt':'#10ba04',
 'reduce_peatland_degradation':'#138d91',
 'peatland_restoration':'#c690f5',
 'reduce_mangrove_conversion':'#71419c',
 'mangrove_restoration':'#51dce0'}

In [11]:
df_long_complete = df_long.copy()
df_long_complete['type'] = df_long_complete['intervention'].map(intervention_type_map)
df_long_complete['group'] = df_long_complete['intervention'].map(ecosystem_type_map)
df_long_complete['color'] = df_long_complete['intervention'].map(color_map)
df_long_complete = df_long_complete[['location', 'intervention', 'color', 'type', 'group', 'value']]

df_long_complete.head()

,location,intervention,color,type,group,value
0,adminregion_ben,reduce_deforestation,#628c5f,protection,non-wetlands,262.9976
1,adminregion_bfa,reduce_deforestation,#628c5f,protection,non-wetlands,115.6229
2,adminregion_cmr,reduce_deforestation,#628c5f,protection,non-wetlands,374.3571
3,adminregion_caf,reduce_deforestation,#628c5f,protection,non-wetlands,262.3109
4,adminregion_tcd,reduce_deforestation,#628c5f,protection,non-wetlands,225.7684


In [12]:
indicator_data_list = []

for location in df_long_complete['location'].unique():
    mitigation_country = df_long_complete[df_long_complete['location'] == location]
    mitigation_country.reset_index(drop=True, inplace=True)
    for i, row in mitigation_country.iterrows():
            
            data_list = []
            for j in range(len(mitigation_country)):
                data_list.append({
                    "id": "mitigation-" + location + "-" + str(j),
                    "type": mitigation_country.iloc[j]['type'],
                    "group": mitigation_country.iloc[j]['group'],
                    "label": mitigation_country.iloc[j]['intervention'],
                    "color": mitigation_country.iloc[j]['color'],
                    "value": mitigation_country.iloc[j]['value'],
                    "format": "number",
                    "unit": "tCO2e/ha",
                })
            
                data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": 'mitigation-' + location,
                "indicator": "wetlands-mitigation-potential",
                "location": location,
                "data": json.loads(data_json),
                "locale":{"en":{"labels":{
                            "reforestation": "Reforestation",
                            "forest_management": "Forest management",
                            "mangrove_restoration": "Mangrove restoration",
                            "peatland_restoration": "Peatland restoration",
                            "reduce_deforestation": "Reduce deforestation",
                            "reduce_mangrove_conversion": "Reduce mangrove conversion",
                            "reduce_peatland_degradation": "Reduce peatland degradation",
                            "grassland_and_savanna_fire_mgmt": "Grassland and savanna fire management"
                            }
                            }
                        }
                }
    indicator_data_list.append(temp_dict)



In [13]:
print(json.dumps(indicator_data_list, indent=2))

[
  {
    "id": "mitigation-adminregion_ben",
    "indicator": "wetlands-mitigation-potential",
    "location": "adminregion_ben",
    "data": [
      {
        "id": "mitigation-adminregion_ben-0",
        "type": "protection",
        "group": "non-wetlands",
        "label": "reduce_deforestation",
        "color": "#628c5f",
        "value": 262.9976,
        "format": "number",
        "unit": "tCO2e/ha"
      },
      {
        "id": "mitigation-adminregion_ben-1",
        "type": "restoration",
        "group": "non-wetlands",
        "label": "reforestation",
        "color": "#98ed93",
        "value": 204.0629,
        "format": "number",
        "unit": "tCO2e/ha"
      },
      {
        "id": "mitigation-adminregion_ben-2",
        "type": "protection",
        "group": "non-wetlands",
        "label": "forest_management",
        "color": "#255c21",
        "value": 3.532,
        "format": "number",
        "unit": "tCO2e/ha"
      },
      {
        "id": "mitigation-ad

### Save data to JSON file

In [14]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "roi-"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('mitigation-')]
# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2, default=lambda x: int(x) if isinstance(x, (np.integer, np.int64)) else x)

## Vector layer

In [ ]:
countries_poly = gpd.read_file('../data/processed/countries_sahel.geojson')
countries_poly.head(3)

,name,bbox,ISO3,geometry
0,Benin,"{'bbox': [0.776667, 6.0398696, 3.8451454, 12.4...",BEN,"POLYGON ((0.77667 10.37667, 0.79220 10.36589, ..."
1,Burkina Faso,"{'bbox': [-5.513207, 9.4104718, 2.4089717, 15....",BFA,"POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
2,Cameroon,"{'bbox': [8.3822176, 1.6517945, 16.1911011, 13...",CMR,"POLYGON ((8.38222 4.34753, 8.39136 4.34017, 8...."


In [ ]:
mitigation_wetlands_df = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/Mitigation_Country_clean.csv')
sel_cols = ['ISO',
        'Reduce deforestation AVG', 'Reforestation AVG', 'Forest management AVG',
       'Grassland and savanna fire mgmt ',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration'] # Selected columns need to be reviewed for final data
mitigation_wetlands_df = mitigation_wetlands_df[sel_cols]
mitigation_wetlands_df.columns = mitigation_wetlands_df.columns.str.strip().str.replace(' ', '_').str.lower().str.replace('_avg', '')
mitigation_wetlands_df.sample(5)


,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
35,BFA,115.6229,199.2904,31.2582,16.8085,NaN,870.0000,NaN,NaN
138,MHL,NaN,547.1500,48.3668,NaN,NaN,NaN,NaN,NaN
218,TWN,NaN,NaN,NaN,NaN,NaN,NaN,1416.6444,704.0
75,FRA,NaN,23.5123,1.5015,NaN,360.9756,758.3871,NaN,NaN
121,LAO,313.7757,169.0422,0.3652,NaN,NaN,1734.8077,NaN,NaN


In [ ]:
mitigation_wetlands_df['wetlands_mitigation_AVG'] = mitigation_wetlands_df[['reduce_peatland_degradation',
                                                                            'peatland_restoration',
                                                                            'reduce_mangrove_conversion',
                                                                            'mangrove_restoration']].mean(axis=1)
mitigation_wetlands_df['other_mitigation_AVG'] = mitigation_wetlands_df[['reduce_deforestation',
                                                                          'reforestation',
                                                                          'forest_management',
                                                                          'grassland_and_savanna_fire_mgmt']].mean(axis=1)
mitigation_wetlands_df.head(3)


,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration,wetlands_mitigation_AVG,other_mitigation_AVG
0,AFG,54.7488,84.1559,100.6039,NaN,NaN,NaN,NaN,NaN,NaN,79.8362
1,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALB,NaN,54.0851,9.6573,NaN,NaN,NaN,NaN,NaN,NaN,31.8712


In [ ]:
mitigation_gdf = mitigation_wetlands_df.merge(countries_poly, left_on='iso', right_on='ISO3')
mitigation_gdf = gpd.GeoDataFrame(mitigation_gdf, geometry='geometry', crs='EPSG:4326')
mitigation_gdf.drop(columns=['bbox', 'ISO3'], inplace=True)
mitigation_gdf.head(3)

,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration,wetlands_mitigation_AVG,other_mitigation_AVG,name,geometry
0,BEN,262.9976,204.0629,3.5320,52.9058,2565.0000,3180.0000,847.0226,704.0,1824.00565,130.874575,Benin,"POLYGON ((0.77667 10.37667, 0.7922 10.36589, 0..."
1,BFA,115.6229,199.2904,31.2582,16.8085,NaN,870.0000,NaN,NaN,870.00000,90.745000,Burkina Faso,"POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
2,CMR,374.3571,218.1430,0.6945,20.0409,1045.2857,1978.6957,1470.9252,704.0,1299.72665,153.308875,Cameroon,"POLYGON ((8.38222 4.34753, 8.39136 4.34017, 8...."


In [ ]:
mitigation_gdf.to_file('../data/processed/mitigation_sahel.geojson', driver='GeoJSON')

In [ ]:
#Create mbtiles
infile = '../data/processed/mitigation_sahel.geojson'
outfile = '../data/processed/mitigation_sahel.mbtiles'
subprocess.run(['tippecanoe', '-zg', '-f', '-o', outfile, infile])